<a href="https://colab.research.google.com/github/jamagiwa/L-Trail/blob/main/reproducibility/Fig1_and_Fig2_pancreas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#L-Trail: Reproducing Figure1 and Figure2 (Pancreas dataset)

In this notebook, we replicate the conceptual framework of Figure 1 and the results for the pancreas dataset from Figure 2.


**Computational Requirements**<br>

Please be aware that the benchmarking section involving scVelo is memory-intensive. Users who wish to focus exclusively on our novel algorithm may skip the scVelo cells to conserve memory and reduce execution time.

In [ ]:
!git clone -b v1.0.0 https://github.com/jamagiwa/L-Trail.git
%cd L-Trail
!pip install -r requirements.txt

In [ ]:
import scvelo as scv
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
from scipy.stats import gaussian_kde, skew
import random
from scipy.stats import skew
from sklearn.neighbors import NearestNeighbors
import seaborn as sns
from scipy.spatial.distance import cosine
from ltrail import pl
from ltrail import tl

In [ ]:
SEED = 34

random.seed(SEED)
np.random.seed(SEED)
sc.settings.seed = SEED
scv.settings.seed = SEED

In [ ]:
#Font settings for publication
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans', 'Bitstream Vera Sans']

# Figure size and font size settings
plt.rcParams['font.size'] = 8
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.labelsize'] = 9
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8
plt.rcParams['legend.fontsize'] = 8
plt.rcParams['figure.titlesize'] = 12

# Scanpy plotting configurations
sc.set_figure_params(
    scanpy=True,
    dpi=100,
    dpi_save=300,
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(6, 6),
    color_map='viridis'
    )

#1. Conceptual Simulation (Figure 1)

A 1D simulation demonstrating the robustness of L-moments against technical outliers compared to conventional central moments.

In [ ]:
# 1D L-moment calculation function
def calc_l_moments_vector_1d(data):
    """
    Computes a directional vector based on L-skewness and L-scale.
    This represents the "L-flow" in a single dimension.
    """
    n = len(data)
    sorted_data = np.sort(data)
    idx = np.arange(1, n + 1)

    # Calculate Probability Weighted Moments (PWMs)
    b0 = np.mean(sorted_data)
    w1 = (idx - 1) / (n - 1)
    b1 = np.mean(sorted_data * w1)
    w2 = (idx - 1) * (idx - 2) / ((n - 1) * (n - 2))
    b2 = np.mean(sorted_data * w2)

    # L-moments
    lambda2 = 2 * b1 - b0
    lambda3 = 6 * b2 - 6 * b1 + b0

    if lambda2 == 0: return 0
    tau3 = lambda3 / lambda2

    # Return the directional vector (Proposed)
    return -tau3 * lambda2

# --- Simulation Setup ---
np.random.seed(34)
n_cells = 200

# Generate main population (Gamma distribution)
main_pop = np.random.gamma(shape=2.0, scale=1.0, size=n_cells)
x = main_pop

# Introduce extreme outliers (simulating technical artifacts)
outliers = np.random.uniform(-10, -8, size=5)
x_corrupted = np.concatenate([x, outliers])

# A. Conventional Approach (Pearson Skewness * SD)
mean_val = np.mean(x_corrupted)
sk_val = skew(x_corrupted)
std_val = np.std(x_corrupted)
vec_pearson = -sk_val * std_val

# B. Proposed Approach (L-moment based)
vec_lflow = calc_l_moments_vector_1d(x_corrupted)

# --- Visualization: Comparative Analysis of Robustness ---
fig, ax = plt.subplots(figsize=(8, 4))

# Histogram (Underlying Distribution)
ax.hist(x_corrupted,
        bins=30,
        density=True,
        alpha=0.5,
        color='gray',
        label='Cell Population (with noise)')

# Plot outliers for visual emphasis
ax.scatter(outliers,
           np.zeros_like(outliers) + 0.01,
           color='red',
           marker='x',
           linewidth=1.5,
           label='Technical Outliers')

# Vertical positioning for the arrows
baseline_y = 0.15

# Arrow A: Conventional Method (Pearson-based)
# Demonstrating high sensitivity to outliers
ax.arrow(mean_val,
         baseline_y,
         vec_pearson,
         0,
         head_width=0.02,
         head_length=0.5,
         fc='red',
         ec='red',
         width=0.005,
         label=f'Conventional (Pearson): Skew={sk_val:.2f}')

# Arrow B: Proposed Method (L-moment/L-flow)
# Demonstrating robustness and stability
ax.arrow(mean_val,
         baseline_y + 0.05,
         vec_lflow,
         0,
         head_width=0.02,
         head_length=0.5,
         fc='blue',
         ec='blue',
         width=0.005,
         label='Proposed (L-moment based)')

# --- Figure Formatting ---
ax.set_title("Sensitivity Comparison: L-moments vs. Conventional Moments", fontsize=14, fontweight='bold')
ax.set_xlabel("Latent Space / Gene Expression (e.g., PC1)", fontsize=12)
ax.set_yticks([]) # Hide Y-axis as it's for density visualization
ax.legend(loc='upper left', fontsize=9, frameon=True)
ax.grid(axis='x', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

#2. Data Acquisiton



Load the pancreas dataset using the scVelo built-in library.


In [ ]:
adata_pc = scv.datasets.pancreas()
adata_pc_velo = adata_pc.copy()

#3. scVelo Benchmark (Stochastic Mode)

Estimate RNA velocity to serve as a kinetic baseline for comparative analysis.

Process the dataset following the standard scVelo preprocessing and velocity estimation pipeline.

In [ ]:
adata_pc_velo = adata_pc.copy()
scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.set_figure_params('scvelo')

scv.pp.filter_genes(adata_pc_velo,
                    min_shared_counts=20)
sc.pp.normalize_total(adata_pc_velo)
sc.pp.log1p(adata_pc_velo)
sc.pp.filter_genes_dispersion(adata_pc_velo, n_top_genes=2000)

scv.pp.moments(adata_pc_velo, n_pcs=30, n_neighbors=30)
scv.tl.velocity(adata_pc_velo, mode='stochastic')
scv.tl.velocity_graph(adata_pc_velo)

In [ ]:
# Explicitly compute PCA and UMAP embeddings
sc.tl.pca(adata_pc_velo, svd_solver='arpack')
sc.tl.umap(adata_pc_velo)

In [ ]:
scv.tl.velocity_graph(adata_pc_velo)

In [ ]:
pc_cellcounts = adata_pc_velo.obs['clusters'].value_counts()
labels = [f"{cluster}\n(n={count})" for cluster, count in zip(pc_cellcounts.index, pc_cellcounts.values)]

fig, ax = plt.subplots(figsize=(15, 6))

sns.barplot(
    x=pc_cellcounts.index,
    y=pc_cellcounts.values,
    order=pc_cellcounts.index,
    color='#4C72B0',
    edgecolor='black',
    linewidth=1,
    ax=ax
)

ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=11)
ax.tick_params(axis='y', labelsize=11)

ax.set_xlabel('Pancreas Clusters', fontsize=14, labelpad=10)
ax.set_ylabel('Number of Cells', fontsize=14, labelpad=10)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

ax.spines['left'].set_linewidth(1)
ax.spines['bottom'].set_linewidth(1)

ax.set_axisbelow(True)
ax.yaxis.grid(True, linestyle='--', color='lightgrey', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
scv.pl.velocity_embedding_stream(adata_pc_velo,
                                 basis='umap',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Pancreas RNA Velocity (Stochastic, umap)',
                                 show=False,
                                 )

scv.pl.velocity_embedding_stream(adata_pc_velo,
                                 basis='pca',
                                 color='clusters',
                                 figsize=(8, 6),
                                 legend_loc='right margin',
                                 title='Pancreas RNA Velocity (Stochastic, pca)',
                                 show=False
                                 )



#4. L-Trail Implementation

Compute and visualize macroscopic transition vectors using L-Trail.

In [ ]:
#pca
pl.plot_ltrail(adata_pc_velo,
            groupby='clusters',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca)'
            )

#umap
pl.plot_ltrail(adata_pc_velo,
            groupby='clusters',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap)'
            )

#5. Quantitative Evaluation of Trajectory Alignment

Local Trajectory Alignment via k-NN Sampling




In [ ]:
tl.calc_velocity_ltrail_similarity(adata_pc_velo,
                                groupby='clusters',
                                method='lmoment',
                                use_rep='X_pca',
                                vel_rep='velocity_pca',
                                )

Compute local cosine similarities between L-Trail vectors and RNA velocity vectors using randomly sampled anchor cells and their local neighborhoods.


In [ ]:
result_knn_cos_lmoment = tl.calc_knn_similarity(adata_pc_velo,
                                             use_rep='X_pca',
                                             vel_rep='velocity_pca',
                                             n_pcs=30,
                                             method='lmoment',
                                             k=50,
                                             n_anchors=1000,
                                             )

pl.plot_knn_similarity(adata_pc_velo,
                    result_knn_cos_lmoment,
                    basis='X_pca',
                    title='Pancreas cos similarity (L-Moment vs Velocity)',
                    figsize=(8, 6),
                    show=True
                    )

pl.boxplot_similarity(adata_pc_velo,
                        result_knn_cos_lmoment,
                        groupby='clusters',
                        figsize=(8, 6),
                        ylim=(-1, 1),
                        title='Pancreas cos similarity (L-Moment vs Velocity)'
                        )

#6. Supplementary Analysis

Evaluate the geometric stability of L-Trail across varying Leiden clustering resolutions (0.3, 0.5, 1.0) and compare it against the conventional Pearson's skewness baseline.

In [ ]:
pc_velo_03 = adata_pc_velo.copy()
pc_velo_05 = adata_pc_velo.copy()
pc_velo_1 = adata_pc_velo.copy()
sc.tl.leiden(pc_velo_03, resolution=0.3)
sc.tl.leiden(pc_velo_05, resolution=0.5)
sc.tl.leiden(pc_velo_1, resolution=1)


pl.plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap, resolution=0.3)'
            )

pl.plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap, resolution=0.5)'
            )

pl.plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, umap, resolution=1)'
            )



In [ ]:
pl.plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, umap, resolution=0.3)'
            )

pl.plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, umap, resolution=0.5)'
            )

pl.plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_umap',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, umap, resolution=1)'
            )

In [ ]:
pl.plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca, resolution=0.3)'
            )

pl.plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca, resolution=0.5)'
            )

pl.plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='lmoment',
            scale=30,
            p_threshold=0.05,
            title='Pancreas L-moment Cluster(scale=30, p<0.05, pca, resolution=1)'
            )



In [ ]:
pl.plot_ltrail(pc_velo_03,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, pca, resolution=0.3)'
            )

pl.plot_ltrail(pc_velo_05,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, pca, resolution=0.5)'
            )

pl.plot_ltrail(pc_velo_1,
            groupby='leiden',
            basis='X_pca',
            use_rep='X_pca',
            method='pearson',
            scale=30,
            p_threshold=0.05,
            title='Pancreas pearson Cluster(scale=30, p<0.05, pca, resolution=1)'
            )